https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews


In [ ]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
print(df.shape)
print(df.columns)
print(df['sentiment'].value_counts())

(50000, 2)
Index(['review', 'sentiment'], dtype='object')
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)   # remove HTML
    text = re.sub(r'http\S+', '', text) # remove URLs
    text = re.sub(r'[^a-zA-Z]', ' ', text) # remove punctuation

    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [stemmer.stem(w) for w in words]

    return " ".join(words)

In [ ]:
df['clean_review'] = df['review'].apply(preprocess_text)

In [ ]:
print("Before:\n", df['review'][0])
print("\nAfter:\n", df['clean_review'][0])

Before:
 One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due t

In [ ]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['clean_review'])
y = df['sentiment']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

### Model Comparison & Insights

- Logistic Regression performed the best as it handles high-dimensional sparse data effectively.
- TF-IDF vectorization improved performance by assigning importance to meaningful words and reducing the impact of common words.
- Naive Bayes was computationally efficient but slightly less accurate due to its assumption of feature independence.
- Decision Tree showed signs of overfitting, as it tends to memorize training data patterns and does not generalize well for unseen text data.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate(y_test, pred, name):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, pred))
    print(classification_report(y_test, pred))

evaluate(y_test, lr_pred, "Logistic Regression")
evaluate(y_test, nb_pred, "Naive Bayes")
evaluate(y_test, dt_pred, "Decision Tree")


Logistic Regression
Accuracy: 0.8863
              precision    recall  f1-score   support

           0       0.90      0.87      0.88      4961
           1       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


Naive Bayes
Accuracy: 0.8508
              precision    recall  f1-score   support

           0       0.86      0.84      0.85      4961
           1       0.85      0.86      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000


Decision Tree
Accuracy: 0.7208
              precision    recall  f1-score   support

           0       0.72      0.72      0.72      4961
           1       0.72      0.72      0.72      5039

    accuracy                           0.72     10000
   macro avg       0.72     

### Model Comparison & Insights

The performance of different models was evaluated using accuracy, precision, recall, and F1-score.

- Logistic Regression achieved the highest accuracy (~88.6%), indicating that it is well-suited for text classification tasks. It effectively handles high-dimensional sparse data generated by TF-IDF.

- Naive Bayes achieved good performance (~85%) and was computationally efficient. However, its assumption of feature independence limits its ability to capture contextual meaning in text.

- Decision Tree showed the lowest accuracy (~72%), indicating poor generalization. It tends to overfit the training data by creating very specific rules, making it less effective for unseen text data.

- TF-IDF vectorization significantly improved performance by assigning importance to meaningful words while reducing the impact of common words.

Overall, Logistic Regression with TF-IDF provided the best balance between accuracy and generalization for this sentiment analysis task.


In [ ]:
import pandas as pd

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'Decision Tree'],
    'Accuracy': [0.8863, 0.8508, 0.7208]
})

print(results)

                 Model  Accuracy
0  Logistic Regression    0.8863
1          Naive Bayes    0.8508
2        Decision Tree    0.7208


### Final Conclusion

This project demonstrated the complete NLP pipeline, including preprocessing, feature engineering using TF-IDF, and model training.

Among the models tested, Logistic Regression performed the best due to its ability to handle sparse and high-dimensional data effectively. Naive Bayes provided a fast baseline, while Decision Tree struggled due to overfitting.

This highlights the importance of choosing the right model and preprocessing techniques in NLP tasks.


## Bag of Words (BoW) Implementation


In [ ]:
# ### Comparison between TF-IDF and BoW

# - TF-IDF performed better than BoW as it considers word importance across documents.
# - BoW only counts word frequency and does not capture importance, leading to slightly lower performance.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Create BoW features
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_review'])

# Target
y_bow = df['sentiment']

In [ ]:
from sklearn.model_selection import train_test_split

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_bow, y_bow, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_bow = LogisticRegression()
lr_bow.fit(Xb_train, yb_train)

yb_pred = lr_bow.predict(Xb_test)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("BoW Logistic Regression Accuracy:", accuracy_score(yb_test, yb_pred))
print(classification_report(yb_test, yb_pred))

BoW Logistic Regression Accuracy: 0.8712
              precision    recall  f1-score   support

           0       0.88      0.86      0.87      4961
           1       0.87      0.88      0.87      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



### BoW vs TF-IDF Comparison

- Bag of Words (BoW) represents text based on word frequency but does not consider word importance across documents.
- TF-IDF improves upon BoW by reducing the weight of common words and giving higher importance to rare and meaningful words.
- In this project, TF-IDF performed better than BoW because it captures the significance of words more effectively.

### NLP Pipeline Flow

Raw Data → Data Understanding → Text Preprocessing → Feature Engineering (TF-IDF & BoW) → Model Training → Evaluation → Comparison → Conclusion

In [ ]:
### NLP Pipeline Flow

Raw Data → Data Understanding → Text Preprocessing → Feature Engineering (TF-IDF & BoW) → Model Training → Evaluation → Comparison → Conclusion